# 03. Character Network and Emotion Analysis

## Objective

The goal of this notebook is to analyze movie characters through two complementary perspectives:

- their interaction structure within the narrative
- their emotional profile based on screenplay text

Rather than analyzing the entire corpus as a single network, this notebook works at the **movie level**, since character interactions only make sense within the boundaries of a specific story.

## Why this matters

This stage moves the project from feature preparation into higher-level analytical modeling.

By combining graph-based interaction patterns with text-based emotional signals, we can begin to understand:

- which characters are structurally central
- which characters dominate the emotional tone of the story
- how narrative importance and emotional behavior relate to each other

## Expected output

By the end of this notebook, we expect to obtain:

- a movie-specific interaction graph
- network metrics for each character
- an initial emotional profile for each character
- a merged analytical table combining structural and emotional features

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import sys

# Add project root to path to find src module
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from src.utils.paths import PROJECT_ROOT

interim_path = PROJECT_ROOT / "data" / "interim" / "character_lines.parquet"
scene_path = PROJECT_ROOT / "data" / "processed" / "scene_dataset.parquet"
character_path = PROJECT_ROOT / "data" / "processed" / "character_dataset.parquet"

# Cargar datasets completos
df_lines = pd.read_parquet(interim_path)
scene_df = pd.read_parquet(scene_path)
character_df = pd.read_parquet(character_path)

# =========================
# Sample complete movies
# =========================
np.random.seed(42)

all_movies = sorted(df_lines["movie_id"].dropna().unique())
n_movies = len(all_movies)

sample_size = max(1, n_movies // 3)   # una tercera parte
sampled_movies = np.random.choice(all_movies, size=sample_size, replace=False)
sampled_movies = sorted(sampled_movies)

# Filtrar cada dataset con las mismas películas
df_lines = df_lines[df_lines["movie_id"].isin(sampled_movies)].copy()
scene_df = scene_df[scene_df["movie_id"].isin(sampled_movies)].copy()
character_df = character_df[character_df["movie_id"].isin(sampled_movies)].copy()

print("Total movies in full dataset:", n_movies)
print("Sampled movies:", len(sampled_movies))
print("df_lines shape:", df_lines.shape)
print("scene_df shape:", scene_df.shape)
print("character_df shape:", character_df.shape)

## 2. Reconstructing cleaned text globally

To support emotion modeling across the full dataset, we create a normalized version of the text.

This cleaning step is intentionally conservative and designed to preserve narrative meaning while reducing noise from OCR and formatting artifacts.

In [4]:
import re

def clean_text(text: str) -> str:
    if not isinstance(text, str):
        return ""

    text = text.replace("’", "'").replace("“", '"').replace("”", '"')
    text = text.replace("'1", "'")
    text = re.sub(r"(?<=[a-zA-Z])1(?=[a-zA-Z])", "", text)
    text = re.sub(r"[^a-zA-Z0-9\s\.\,\!\?\']", " ", text)
    text = text.lower()
    text = re.sub(r"\s+", " ", text).strip()

    return text

df_lines["clean_text"] = df_lines["text"].apply(clean_text)

df_lines[["text", "clean_text"]].head()

,text,clean_text
2250,"A very hot summer afternoon. It is a large, dr...","a very hot summer afternoon. it is a large, dr..."
2251,"Well, I figured we might want to vote by ballots.","well, i figured we might want to vote by ballots."
2252,"Well, I was figuring we'd take a five-minute b...","well, i was figuring we'd take a five minute b..."
2253,"All right, gentlemen. Let's take seats.","all right, gentlemen. let's take seats."
2254,"Well, I was thinking we ought to sit in order,...","well, i was thinking we ought to sit in order,..."


## 3. Filtering dialogue lines

Emotion modeling is applied only to dialogue entries, since descriptive text does not directly represent a character’s spoken emotional expression.

This step reduces the volume of data to the subset most relevant for emotional analysis.

In [5]:
df_dialog = df_lines[df_lines["label"] == "dialog"].copy()

print("df_dialog shape:", df_dialog.shape)
print("Unique movies in df_dialog:", df_dialog["movie_id"].nunique())
df_dialog.head()

df_dialog shape: (349300, 12)
Unique movies in df_dialog: 717


,movie_id,character_name,source_file,segment_id,scene_id,scene_key,label,text,word_count,text_length,line_number,clean_text
2251,12 Angry Men_0118528,Foreman,/Users/jesussalgado/Downloads/movie-character-...,4,11,4_11,dialog,"Well, I figured we might want to vote by ballots.",10,49,2,"well, i figured we might want to vote by ballots."
2252,12 Angry Men_0118528,Foreman,/Users/jesussalgado/Downloads/movie-character-...,4,24,4_24,dialog,"Well, I was figuring we'd take a five-minute b...",16,88,3,"well, i was figuring we'd take a five minute b..."
2253,12 Angry Men_0118528,Foreman,/Users/jesussalgado/Downloads/movie-character-...,4,44,4_44,dialog,"All right, gentlemen. Let's take seats.",6,39,4,"all right, gentlemen. let's take seats."
2254,12 Angry Men_0118528,Foreman,/Users/jesussalgado/Downloads/movie-character-...,4,48,4_48,dialog,"Well, I was thinking we ought to sit in order,...",30,156,5,"well, i was thinking we ought to sit in order,..."
2255,12 Angry Men_0118528,Foreman,/Users/jesussalgado/Downloads/movie-character-...,4,60,4_60,dialog,(to the 8th Juror) How about sitting down? (th...,30,169,6,to the 8th juror how about sitting down? the 8...


## 4. Loading the emotion classification model

We use a transformer-based emotion classifier to assign dialogue-level emotion scores.

This model provides a richer and more nuanced emotional representation than simple keyword-based methods.

In [6]:
from transformers import pipeline

emotion_pipeline = pipeline(
    "text-classification",
    model="j-hartmann/emotion-english-distilroberta-base",
    top_k=None
)

EMOTION_LABELS = ["anger", "disgust", "fear", "sadness", "neutral", "surprise", "joy"]

def scores_to_row(prediction_list):
    score_map = {item["label"]: item["score"] for item in prediction_list}
    return {label: score_map.get(label, 0.0) for label in EMOTION_LABELS}

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 105/105 [00:00<00:00, 26488.75it/s]
RobertaForSequenceClassification LOAD REPORT from: j-hartmann/emotion-english-distilroberta-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## 5. Running batch emotion inference across all movies

Since the dialogue dataset is large, we process it in batches rather than in a single pass.

This approach is more stable, easier to debug, and allows the notebook to save progress incrementally if needed.

In [7]:
from math import ceil

BATCH_SIZE = 128

all_rows = []
texts = df_dialog["clean_text"].tolist()
n_batches = ceil(len(texts) / BATCH_SIZE)

for batch_idx in range(n_batches):
    start = batch_idx * BATCH_SIZE
    end = min((batch_idx + 1) * BATCH_SIZE, len(texts))
    batch_texts = texts[start:end]

    preds = emotion_pipeline(batch_texts, truncation=True)
    batch_rows = [scores_to_row(pred) for pred in preds]
    all_rows.extend(batch_rows)

    if batch_idx % 100 == 0:
        print(f"Processed batch {batch_idx + 1}/{n_batches}")

emotion_df = pd.DataFrame(all_rows)

dialogue_emotion_df = pd.concat(
    [df_dialog.reset_index(drop=True), emotion_df.reset_index(drop=True)],
    axis=1
)

print("dialogue_emotion_df shape:", dialogue_emotion_df.shape)
dialogue_emotion_df.head()

Processed batch 1/2729
Processed batch 101/2729
Processed batch 201/2729
Processed batch 301/2729
Processed batch 401/2729
Processed batch 501/2729
Processed batch 601/2729
Processed batch 701/2729
Processed batch 801/2729
Processed batch 901/2729
Processed batch 1001/2729
Processed batch 1101/2729
Processed batch 1201/2729
Processed batch 1301/2729
Processed batch 1401/2729
Processed batch 1501/2729
Processed batch 1601/2729
Processed batch 1701/2729
Processed batch 1801/2729
Processed batch 1901/2729
Processed batch 2001/2729
Processed batch 2101/2729
Processed batch 2201/2729
Processed batch 2301/2729
Processed batch 2401/2729
Processed batch 2501/2729
Processed batch 2601/2729
Processed batch 2701/2729
dialogue_emotion_df shape: (349300, 19)


,movie_id,character_name,source_file,segment_id,scene_id,scene_key,label,text,word_count,text_length,line_number,clean_text,anger,disgust,fear,sadness,neutral,surprise,joy
0,12 Angry Men_0118528,Foreman,/Users/jesussalgado/Downloads/movie-character-...,4,11,4_11,dialog,"Well, I figured we might want to vote by ballots.",10,49,2,"well, i figured we might want to vote by ballots.",0.006751,0.006855,0.001613,0.007816,0.933125,0.022964,0.020877
1,12 Angry Men_0118528,Foreman,/Users/jesussalgado/Downloads/movie-character-...,4,24,4_24,dialog,"Well, I was figuring we'd take a five-minute b...",16,88,3,"well, i was figuring we'd take a five minute b...",0.041554,0.491412,0.014553,0.028873,0.233993,0.184641,0.004973
2,12 Angry Men_0118528,Foreman,/Users/jesussalgado/Downloads/movie-character-...,4,44,4_44,dialog,"All right, gentlemen. Let's take seats.",6,39,4,"all right, gentlemen. let's take seats.",0.064805,0.040231,0.008010,0.012112,0.789938,0.002503,0.082402
3,12 Angry Men_0118528,Foreman,/Users/jesussalgado/Downloads/movie-character-...,4,48,4_48,dialog,"Well, I was thinking we ought to sit in order,...",30,156,5,"well, i was thinking we ought to sit in order,...",0.009432,0.009175,0.002047,0.005419,0.958988,0.003787,0.011151
4,12 Angry Men_0118528,Foreman,/Users/jesussalgado/Downloads/movie-character-...,4,60,4_60,dialog,(to the 8th Juror) How about sitting down? (th...,30,169,6,to the 8th juror how about sitting down? the 8...,0.015911,0.001652,0.800245,0.002094,0.007614,0.170368,0.002116


## 6. Building character-level emotional profiles

Once dialogue-level emotions are available, we aggregate them by movie and character.

This gives each character a stable emotional profile that can later be merged with graph and narrative features.

In [8]:
emotion_by_character = (
    dialogue_emotion_df
    .groupby(["movie_id", "character_name"])[EMOTION_LABELS]
    .mean()
    .reset_index()
)

print("emotion_by_character shape:", emotion_by_character.shape)
emotion_by_character.head()

emotion_by_character shape: (11081, 9)


,movie_id,character_name,anger,disgust,fear,sadness,neutral,surprise,joy
0,12 Angry Men_0118528,Foreman,0.095498,0.044350,0.029834,0.073354,0.458029,0.218354,0.080580
1,12 Angry Men_0118528,Guard,0.042017,0.021202,0.214416,0.122547,0.417218,0.142367,0.040234
2,12 Angry Men_0118528,Judge,0.082595,0.002333,0.011364,0.860545,0.020602,0.014094,0.008468
3,2012_1190080,Adrian Helmsley,0.057704,0.039261,0.032216,0.134177,0.397875,0.232143,0.106624
4,2012_1190080,Alec,0.054361,0.017811,0.051260,0.309135,0.307316,0.234196,0.025922


## 7. Building movie-specific interaction features

Character interaction graphs are computed separately for each movie, since interactions are only meaningful within a shared narrative.

For each movie, we construct weighted co-occurrence edges and derive graph-based metrics such as:

- degree centrality
- weighted degree
- betweenness

In [9]:
import networkx as nx
from itertools import combinations

scene_df = (
    df_lines.groupby(["movie_id", "segment_id"])
    .agg(
        characters_present=("character_name", lambda x: sorted(set(x)))
    )
    .reset_index()
)

graph_feature_rows = []

for movie_id, movie_scenes in scene_df.groupby("movie_id"):
    edges = []

    for _, row in movie_scenes.iterrows():
        chars = row["characters_present"]
        if len(chars) < 2:
            continue
        for c1, c2 in combinations(chars, 2):
            edges.append((c1, c2))

    if not edges:
        continue

    edges_df = pd.DataFrame(edges, columns=["char_1", "char_2"])
    edges_weighted = (
        edges_df.groupby(["char_1", "char_2"])
        .size()
        .reset_index(name="weight")
    )

    G = nx.Graph()
    for _, row in edges_weighted.iterrows():
        G.add_edge(row["char_1"], row["char_2"], weight=row["weight"])

    degree_centrality = nx.degree_centrality(G)
    weighted_degree = dict(G.degree(weight="weight"))
    betweenness = nx.betweenness_centrality(G)

    for node in G.nodes():
        graph_feature_rows.append({
            "movie_id": movie_id,
            "character_name": node,
            "degree_centrality": degree_centrality.get(node, 0.0),
            "weighted_degree": weighted_degree.get(node, 0.0),
            "betweenness": betweenness.get(node, 0.0)
        })

graph_features_df = pd.DataFrame(graph_feature_rows)

print("graph_features_df shape:", graph_features_df.shape)
graph_features_df.head()

graph_features_df shape: (10756, 5)


,movie_id,character_name,degree_centrality,weighted_degree,betweenness
0,12 Angry Men_0118528,Foreman,0.500000,2,0.000000
1,12 Angry Men_0118528,Guard,1.000000,3,1.000000
2,12 Angry Men_0118528,Judge,0.500000,1,0.000000
3,2012_1190080,Adrian Helmsley,0.620690,113,0.287682
4,2012_1190080,Boxing Announcer,0.241379,7,0.027609


## 9. Merging narrative, graph, and emotional features

At this stage, we combine all character-level components into a single analytical dataset.

This final dataset integrates:

- narrative importance
- graph-based structural features
- emotional profiles

In [10]:
final_character_df = (
    character_df
    .merge(graph_features_df, on=["movie_id", "character_name"], how="left")
    .merge(emotion_by_character, on=["movie_id", "character_name"], how="left")
)

print("final_character_df shape:", final_character_df.shape)
final_character_df.head()

final_character_df shape: (11081, 19)


,movie_id,character_name,total_lines,total_scenes,total_words,avg_words_per_line,avg_text_length,importance_score,rank_in_movie,degree_centrality,weighted_degree,betweenness,anger,disgust,fear,sadness,neutral,surprise,joy
0,Margaret_0466893,Lisa Cohen,844,844,13522,16.021327,84.694313,681.9610,1.0,1.00,142.0,0.275253,0.096101,0.049202,0.041899,0.099627,0.311601,0.293370,0.108201
1,Crazylove_0416658,Letty Mayer,844,844,9660,11.445498,62.534360,680.0300,1.0,0.88,177.0,0.560349,0.087902,0.043099,0.041384,0.169019,0.343337,0.189704,0.125555
2,The Woodsman_0361127,Walter,794,794,8807,11.091940,59.668766,639.6035,1.0,1.00,64.0,0.696429,0.097426,0.067469,0.049352,0.138883,0.320983,0.221781,0.104106
3,500 Days of Summer_1022603,Tom,737,737,8962,12.160109,64.949796,594.0810,1.0,1.00,52.0,0.045299,0.096406,0.046319,0.029407,0.123579,0.277247,0.272397,0.154644
4,The Wolf of Wall Street_0993846,Jordan Belfort,723,723,14333,19.824343,116.362379,585.5665,1.0,1.00,217.0,0.496725,0.186026,0.058900,0.035905,0.095675,0.273239,0.222826,0.127429


## 10. Clustering characters across movies

To discover reusable character archetypes across multiple narratives, we cluster characters using their combined narrative, structural, and emotional features.

Unlike graph features, which are computed within movies, clustering is applied globally to identify broader patterns across the corpus.

In [11]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

feature_cols = [
    "importance_score",
    "degree_centrality",
    "weighted_degree",
    "betweenness",
    "anger",
    "disgust",
    "fear",
    "sadness",
    "neutral",
    "surprise",
    "joy"
]

X = final_character_df[feature_cols].fillna(0)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
final_character_df["cluster"] = kmeans.fit_predict(X_scaled)

final_character_df[["movie_id", "character_name", "cluster"]].head()

,movie_id,character_name,cluster
0,Margaret_0466893,Lisa Cohen,4
1,Crazylove_0416658,Letty Mayer,4
2,The Woodsman_0361127,Walter,4
3,500 Days of Summer_1022603,Tom,4
4,The Wolf of Wall Street_0993846,Jordan Belfort,4


## 11. Saving processed outputs

Finally, we save the two main artifacts needed for downstream analysis and dashboard development:

- `dialogue_emotion_dataset.parquet`
- `final_character_dataset.parquet`

In [13]:
processed_dir = project_root / "data" / "processed"
dialogue_emotion_df.to_parquet(
    processed_dir / "dialogue_emotion_dataset.parquet",
    index=False
)

final_character_df.to_parquet(
    processed_dir / "final_character_dataset.parquet",
    index=False
)

print("Saved dialogue_emotion_dataset.parquet:", dialogue_emotion_df.shape)
print("Saved final_character_dataset.parquet:", final_character_df.shape)

Saved dialogue_emotion_dataset.parquet: (349300, 19)
Saved final_character_dataset.parquet: (11081, 20)
